In [153]:
import yfinance as yf
import pandas as pd
import numpy as np
import pandas_ta as ta
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils import class_weight

# For backtesting
from backtesting import Backtest, Strategy
from backtesting.lib import crossover

In [154]:
def download_crypto_data(ticker, period="2y", interval="1h"):
    data = yf.download(ticker, period=period, interval=interval)
    
    # If data has multi-index columns, drop the second level (ticker name)
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)  # Keep only "Open", "High", etc.

    data.dropna(inplace=True)
    return data

# Example: Download Bitcoin (BTC-USD) data
# btc_data = download_crypto_data("BTC-USD")
# print(btc_data)

In [155]:
def add_technical_indicators(df):
    # Example technical indicators (avoid candlestick pattern indicators)
    df['RSI'] = ta.rsi(df['Close'], length=14)
    macd = ta.macd(df['Close'])
    # print(df)
    # print(macd)
    df = df.join(macd)
    df['SMA_50'] = ta.sma(df['Close'], length=50)
    df['SMA_200'] = ta.sma(df['Close'], length=200)
    bbands = ta.bbands(df['Close'], length=20, std=2)
    df = df.join(bbands)  # Columns: BBL_20_2.0, BBM_20_2.0, BBU_20_2.0
    stoch = ta.stoch(df['High'], df['Low'], df['Close'])
    df = df.join(stoch)  # Typically STOCHk_14_3_3 and STOCHd_14_3_3
    adx_df = ta.adx(df['High'], df['Low'], df['Close'])
    df['ADX'] = adx_df['ADX_14']  # Using ADX_14 for simplicity
    df['OBV'] = ta.obv(df['Close'], df['Volume'])
    
    # Remove any rows with missing indicator values
    df.dropna(inplace=True)
    return df

# btc_data = add_technical_indicators(btc_data)

In [156]:
def define_target(df):
    """
    Modify target to predict the percent change in stock price for the next period.
    """
    df = df.copy()
    df['Return'] = df['Close'].pct_change().shift(-1)
    df.dropna(inplace=True)
    return df

# btc_data = define_target(btc_data)

In [157]:
# Download data
# btc_data = download_crypto_data("BTC-USD")
# btc_data = add_technical_indicators(btc_data)

cryptos = ['BTC-USD', 'ETH-USD', 'BNB-USD', 'SOL-USD', 'XRP-USD']
# Prepare features and target arrays
features = [
    'RSI', 
    'MACD_12_26_9', 'MACDh_12_26_9', 'MACDs_12_26_9', 
    'SMA_50', 'SMA_200',
    'BBL_20_2.0', 'BBM_20_2.0', 'BBU_20_2.0',
    'STOCHk_14_3_3', 'STOCHd_14_3_3',
    'ADX', 'OBV', 'Close'
]

# Prepare data for multiple cryptocurrencies
all_X, all_y = [], []
scaler_X, scaler_y = StandardScaler(), StandardScaler()
time_steps = 60

In [158]:
def create_sequences(data, time_steps):
    sequences = []
    for i in range(len(data) - time_steps):
        sequences.append(data[i:(i + time_steps)])
    return np.array(sequences)

for crypto in cryptos:
    df = download_crypto_data(crypto)
    df = add_technical_indicators(df)
    df = define_target(df)

    X = df[features].values
    y = df['Return'].values.reshape(-1, 1)

    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y)

    split = int(0.8 * len(X_scaled))
    X_train, X_test = X_scaled[:split], X_scaled[split:]
    y_train, y_test = y_scaled[:split], y_scaled[split:]

    X_train_seq = create_sequences(X_train, time_steps)
    X_test_seq = create_sequences(X_test, time_steps)
    y_train_seq = y_train[time_steps:]
    y_test_seq = y_test[time_steps:]

    y_train_seq = y_train_seq[:X_train_seq.shape[0]]
    y_test_seq = y_test_seq[:X_test_seq.shape[0]]

    all_X.append(X_train_seq)
    all_y.append(y_train_seq)

X_train_final = np.concatenate(all_X, axis=0)
y_train_final = np.concatenate(all_y, axis=0)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [159]:
# Define the LSTM model
model = Sequential([
    Bidirectional(LSTM(100, return_sequences=True, input_shape=(X_train_seq.shape[1], X_train_seq.shape[2]))),
    Dropout(0.3),
    LSTM(50, return_sequences=True),
    Dropout(0.3),
    LSTM(25, return_sequences=False),
    Dropout(0.3),
    Dense(50, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='linear')
])

model.compile(optimizer=Adam(learning_rate=0.001), 
              loss='mse', 
              metrics=['mae'])

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train model
history = model.fit(X_train_seq, y_train_seq, 
                    epochs=100, 
                    batch_size=32, 
                    validation_split=0.2, 
                    callbacks=[early_stop],
                    verbose=1)

loss, accuracy = model.evaluate(X_test_seq, y_test_seq, verbose=0)
print(f"Test Accuracy: {accuracy:.4f}")

Epoch 1/100


c:\Users\cdpea\miniforge3\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


344/344 ━━━━━━━━━━━━━━━━━━━━ 26s 53ms/step - loss: 0.7456 - mae: 0.4554 - val_loss: 0.6351 - val_mae: 0.4798
Epoch 2/100
344/344 ━━━━━━━━━━━━━━━━━━━━ 17s 48ms/step - loss: 0.6393 - mae: 0.4364 - val_loss: 0.6349 - val_mae: 0.4799
Epoch 3/100
344/344 ━━━━━━━━━━━━━━━━━━━━ 16s 48ms/step - loss: 0.5665 - mae: 0.4372 - val_loss: 0.6356 - val_mae: 0.4808
Epoch 4/100
344/344 ━━━━━━━━━━━━━━━━━━━━ 16s 48ms/step - loss: 0.5113 - mae: 0.4320 - val_loss: 0.6346 - val_mae: 0.4798
Epoch 5/100
344/344 ━━━━━━━━━━━━━━━━━━━━ 16s 48ms/step - loss: 0.6134 - mae: 0.4414 - val_loss: 0.6350 - val_mae: 0.4799
Epoch 6/100
344/344 ━━━━━━━━━━━━━━━━━━━━ 16s 48ms/step - loss: 0.5752 - mae: 0.4307 - val_loss: 0.6349 - val_mae: 0.4806
Epoch 7/100
344/344 ━━━━━━━━━━━━━━━━━━━━ 16s 48ms/step - loss: 0.5907 - mae: 0.4350 - val_loss: 0.6347 - val_mae: 0.4806
Epoch 8/100
344/344 ━━━━━━━━━━━━━━━━━━━━ 17s 48ms/step - loss: 0.7369 - mae: 0.4420 - val_loss: 0.6348 - val_mae: 0.4801
Epoch 9/100
344/344 ━━━━━━━━━━━━━━━━━━━━ 17s

In [164]:
def prepare_backtest_data(df, model, scaler_X, scaler_y, time_steps=60):
    df = df.copy()
    df = add_technical_indicators(df)
    print(len(df))

    X_features = df[features].values
    X_scaled = scaler_X.transform(X_features)

    X_seq = create_sequences(X_scaled, time_steps)

    preds = model.predict(X_seq)
    preds = scaler_y.inverse_transform(preds)
    print(len(preds))

    df = df[time_steps:]
    df['PredictedChange'] = preds
    df['Signal'] = np.where(df['PredictedChange'] >= 0, 1, -1)

    return df

# Define backtest strategy
class NeuralNetStrategy(Strategy):
    def init(self):
        self.signal = self.data.Signal

    def next(self):
        current_signal = self.signal[-1]
        if current_signal == 1:
            if self.position:
                self.position.close()
            self.buy()
        elif current_signal == -1:
            if self.position:
                self.position.close()
            self.sell()

btc_data = download_crypto_data("BTC-USD")
bt_data = prepare_backtest_data(btc_data, model, scaler_X, scaler_y)

bt = Backtest(bt_data, NeuralNetStrategy, cash=1000000, commission=0.002, exclusive_orders=True)
stats = bt.run()
print(stats)
bt.plot()

[*********************100%***********************]  1 of 1 completed


17258
538/538 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step
17198


Backtest.run:   0%|          | 0/17197 [00:00<?, ?bar/s]c:\Users\cdpea\miniforge3\Lib\site-packages\backtesting\backtesting.py:954: UserWarning: time=1023: Broker canceled the relative-sized order due to insufficient margin.
  warnings.warn(
c:\Users\cdpea\miniforge3\Lib\site-packages\backtesting\backtesting.py:954: UserWarning: time=1024: Broker canceled the relative-sized order due to insufficient margin.
  warnings.warn(
c:\Users\cdpea\miniforge3\Lib\site-packages\backtesting\backtesting.py:954: UserWarning: time=1025: Broker canceled the relative-sized order due to insufficient margin.
  warnings.warn(
c:\Users\cdpea\miniforge3\Lib\site-packages\backtesting\backtesting.py:954: UserWarning: time=1026: Broker canceled the relative-sized order due to insufficient margin.
  warnings.warn(
c:\Users\cdpea\miniforge3\Lib\site-packages\backtesting\backtesting.py:954: UserWarning: time=1027: Broker canceled the relative-sized order due to insufficient margin.
  warnings.warn(
c:\Users\cdpea

Start                     2023-04-12 00:00...
End                       2025-04-01 05:00...
Duration                    720 days 05:00:00
Exposure Time [%]                     6.16351
Equity Final [$]                  24873.59686
Equity Peak [$]                     1000000.0
Commissions [$]                  946389.90509
Return [%]                          -97.51264
Buy & Hold Return [%]               175.40225
Return (Ann.) [%]                   -83.87019
Volatility (Ann.) [%]                 6.19051
CAGR [%]                            -84.61973
Sharpe Ratio                        -13.54818
Sortino Ratio                        -2.20864
Calmar Ratio                          -0.8601
Alpha [%]                          -106.08014
Beta                                  0.04884
Max. Drawdown [%]                   -97.51264
Avg. Drawdown [%]                   -97.51264
Max. Drawdown Duration      720 days 04:00:00
Avg. Drawdown Duration      720 days 04:00:00
# Trades                          

GridPlot(id='p3952', ...)